<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/903_MSOv3_NodeTests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Your **node test suite is very well designed**. It shows a good understanding of how LangGraph-style agents should be tested: **logic first (utilities), integration second (nodes), orchestration last (graph)**.

What you’re doing here is effectively **integration testing between utilities and state**, which is exactly the purpose of node tests.

I'll break this down into:

1️⃣ What you're doing right
2️⃣ One small improvement to test robustness
3️⃣ Two high-value tests worth adding
4️⃣ Why this testing structure is ideal for agent systems

---

# 1️⃣ What You're Doing Exactly Right

Your tests correctly verify that **nodes act as thin wrappers around utilities**.

Your node structure:

```
load_node → telemetry utility
metrics_node → metrics utility
report_node → reporting utilities
```

And your tests reflect that.

You are validating that nodes:

```
accept state
call utility
update state
return updated state
```

That’s the exact behavior nodes should have.

---

# 2️⃣ Load Node Test — Excellent

This test is strong:

```python
test_load_node_with_temp_data
```

You correctly simulate telemetry files using `tmp_path`.

You verify:

```
mission_runs loaded
run_count updated
errors preserved
```

Which confirms the node correctly delegates to `load_telemetry`.

Very good.

---

# 3️⃣ Missing Project Root Test — Smart

This test is subtle but important:

```python
test_load_node_missing_project_root_uses_cwd
```

You're verifying that the node **doesn't crash if project_root isn't provided**.

That’s important because:

```
production pipelines often omit optional state fields
```

Your assertion:

```
assert isinstance(out.get("mission_runs"), list)
```

is also good practice because you're testing **behavior not exact output**.

---

# 4️⃣ Metrics Node Test — Well Scoped

This test confirms the node correctly computes signals:

```python
test_metrics_node_updates_state
```

You verify:

```
mission_status_signal
data_quality_signal
metrics_available
```

These are exactly the outputs that matter.

You also wisely avoid asserting exact numeric values, which keeps tests resilient.

---

# 5️⃣ Report Node File Test — Very Good

This test is essential:

```python
test_report_node_writes_to_temp_dir
```

You're verifying:

```
report_file_path exists
file created
content written
```

Which confirms the **final pipeline step works**.

Your use of `tmp_path` keeps the test isolated.

That’s exactly how file-writing tests should be done.

---

# 6️⃣ Error Handling Test — Excellent

This test is particularly good:

```python
test_report_node_on_error_appends_errors
```

You intentionally give **incomplete state**:

```python
state = {"errors": [], "mission_id": "M001"}
```

Then verify:

```
node doesn't crash
errors list exists
error recorded if failure occurs
```

This confirms the node is **defensive**.

Which is very important for orchestrators.

---

# Small Improvement I Recommend

Your path injection works:

```python
_root = Path(__file__).resolve().parent
sys.path.insert(0, str(_root))
```

But it’s slightly better to resolve **project root instead of test folder**.

Example:

```python
PROJECT_ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))
```

Because most projects are structured like:

```
project/
  agents/
  config.py
  tests/
```

Using the project root ensures imports always resolve correctly.

This is optional but cleaner.

---

# Two Additional Node Tests I Recommend

Your coverage is already good, but two tests would strengthen it further.

---

# Test 1: Mission Filtering Through Node

You tested filtering in utilities.

But you should confirm the **node preserves it**.

Example:

```python
def test_load_node_filters_runs_by_mission(tmp_path):
    (tmp_path / "mission_runs.json").write_text(
        '[{"run_id":"R1","mission_id":"M001"},{"run_id":"R2","mission_id":"M002"}]',
        encoding="utf-8"
    )
    (tmp_path / "task_execution_logs.json").write_text("[]", encoding="utf-8")
    (tmp_path / "mission_risk_events.json").write_text("[]", encoding="utf-8")
    (tmp_path / "agent_performance_stats.json").write_text("[]", encoding="utf-8")

    config = MissionOrchestratorV3Config(data_dir=str(tmp_path))
    load_node = nodes.make_load_node(config)

    state = {"mission_id": "M001", "project_root": str(tmp_path), "errors": []}
    out = load_node(state)

    assert len(out["mission_runs"]) == 1
    assert out["mission_runs"][0]["mission_id"] == "M001"
```

This confirms **node + utility integration**.

---

# Test 2: Report Node With Recommendation Logic

You recently added recommendation logic, which is important.

So we should ensure the node generates it.

Example:

```python
def test_report_node_includes_recommendation(tmp_path):
    config = MissionOrchestratorV3Config(reports_dir=str(tmp_path))
    report_node = nodes.make_report_node(config)

    state = {
        "mission_id": "M001",
        "project_root": str(tmp_path),
        "errors": [],
        "mission_runs": [{"kpis": {"onboarding_time_days": 2.1}}],
        "mission_status_signal": "AT RISK",
        "mission_status_reason": "High severity risk events detected",
        "data_quality_signal": "HIGH",
        "data_quality_details": "1 runs",
        "mission_name": "Test Mission",
        "telemetry_summary": {"runs_loaded": 1, "tasks_loaded": 0, "risk_events_loaded": 1},
        "mission_risk_events": [{"severity": "HIGH", "risk_type": "approval_delay"}],
    }

    out = report_node(state)

    report = Path(out["report_file_path"]).read_text()
    assert "Recommendation:" in report
```

This ensures your **most important report enhancement never breaks**.

---

# 4️⃣ Why Your Testing Strategy Is Excellent

Your test layers now look like this:

```
UTILITIES TESTS
  telemetry
  metrics
  reporting

NODE TESTS
  load node
  metrics node
  report node

GRAPH
  implicitly tested through nodes
```

That structure is ideal.

Testing the graph directly is rarely necessary because the graph is just:

```
routing logic
```

Your **actual logic lives in utilities**.

Which you already tested.

---

# Final Verdict

Your testing approach is **very strong and production-quality**.

You are verifying:

```
data ingestion
state mutation
signal generation
report generation
error resilience
file writing
```

Which are the **critical surfaces of an orchestrator**.

With the two extra tests I suggested, your MSO testing suite becomes **excellent**.



In [ ]:
"""
Node tests for MSO v3: load, metrics, report nodes with mock state and temp dirs.
Run from project root: python -m pytest test_mso_v3_nodes.py -v --tb=short
"""
import sys
from pathlib import Path

_root = Path(__file__).resolve().parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import pytest

from config import MissionOrchestratorV3Config
from agents.mso_v3.orchestrator import nodes


def test_load_node_with_temp_data(tmp_path):
    """Load node populates state from JSON in data dir."""
    (tmp_path / "mission_runs.json").write_text('[{"run_id": "R1", "mission_id": "M001"}]', encoding="utf-8")
    (tmp_path / "task_execution_logs.json").write_text("[]", encoding="utf-8")
    (tmp_path / "mission_risk_events.json").write_text("[]", encoding="utf-8")
    (tmp_path / "agent_performance_stats.json").write_text("[]", encoding="utf-8")
    (tmp_path / "mission_kpis.json").write_text("{}", encoding="utf-8")

    config = MissionOrchestratorV3Config(data_dir=str(tmp_path), reports_dir=str(tmp_path / "out"))
    load_node = nodes.make_load_node(config)
    state = {"mission_id": "M001", "project_root": str(tmp_path), "errors": []}
    out = load_node(state)
    assert out.get("errors") == []
    assert len(out.get("mission_runs", [])) == 1
    assert out.get("run_count") == 1


def test_load_node_missing_project_root_uses_cwd():
    """Load node runs without project_root (uses default resolution)."""
    config = MissionOrchestratorV3Config()
    load_node = nodes.make_load_node(config)
    state = {"mission_id": "M001", "errors": []}
    out = load_node(state)
    assert "mission_runs" in out
    assert "run_count" in out
    # Without real data dir, runs may be 0; we only check no crash and keys present
    assert isinstance(out.get("mission_runs"), list)


def test_metrics_node_updates_state():
    """Metrics node adds signature metrics and signals to state."""
    config = MissionOrchestratorV3Config()
    metrics_node = nodes.make_metrics_node(config)
    state = {
        "mission_runs": [{"mission_id": "M001", "total_duration_minutes": 50, "human_interventions": 0, "kpis": {"onboarding_time_days": 2.0}}],
        "task_execution_logs": [],
        "mission_risk_events": [],
        "agent_performance_stats": [],
        "mission_kpis": {},
        "run_count": 1,
    }
    out = metrics_node(state)
    assert out.get("errors", []) == []  # metrics node may not return errors key (merge preserves prior)
    assert "mission_status_signal" in out
    assert "data_quality_signal" in out
    assert "metrics_available" in out


def test_report_node_writes_to_temp_dir(tmp_path):
    """Report node writes markdown file to reports_dir."""
    config = MissionOrchestratorV3Config(reports_dir=str(tmp_path / "reports"))
    report_node = nodes.make_report_node(config)
    state = {
        "mission_id": "M001",
        "project_root": str(tmp_path),
        "errors": [],
        "mission_runs": [{"mission_id": "M001", "kpis": {"onboarding_time_days": 2.1}}],
        "mission_status_signal": "HEALTHY",
        "mission_status_reason": "All metrics within acceptable range.",
        "data_quality_signal": "HIGH",
        "data_quality_details": "1 runs",
        "mission_name": "Test Mission",
        "telemetry_summary": {"runs_loaded": 1, "tasks_loaded": 0, "risk_events_loaded": 0},
        "task_execution_logs": [],
        "mission_risk_events": [],
        "agent_performance_stats": [],
        "mission_portfolio_snapshot": None,
        "mission_kpis": {},
    }
    out = report_node(state)
    assert out.get("errors") == []
    assert out.get("report_file_path")
    assert Path(out["report_file_path"]).exists()
    assert "MISSION STATUS" in Path(out["report_file_path"]).read_text(encoding="utf-8")


def test_report_node_on_error_appends_errors():
    """Report node catches exception and appends to state errors."""
    config = MissionOrchestratorV3Config()
    report_node = nodes.make_report_node(config)
    state = {"errors": [], "mission_id": "M001"}  # missing required keys for full report
    out = report_node(state)
    # Node may still succeed if build_report_sections/assemble_report handle missing keys
    assert "errors" in out
    # If it failed, we have at least one error
    if out.get("report_file_path"):
        assert Path(out["report_file_path"]).exists()
    else:
        assert len(out["errors"]) >= 1
